# Figures: Viasat validation

Notebook for the paper validation section. The figures compare the 2020 and 2021 Viasat campaigns through the penetration ratio `eta` already computed by the pipeline, and are saved in `article/results/figures/validazione/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.stats import ks_2samp, pearsonr

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'article' / 'results').exists():
            return candidate
    raise FileNotFoundError('Run the notebook inside the covid_hep repository.')

ROOT = find_root()
RESULTS = ROOT / 'article' / 'results'
OUT = RESULTS / 'figures' / 'validazione'
OUT.mkdir(parents=True, exist_ok=True)
PALETTE = {'2020': '#0072B2', '2021': '#D55E00'}
sns.set_theme(context='talk', style='ticks', font_scale=1.25)
plt.rcParams.update({'font.size': 18, 'axes.titlesize': 22, 'axes.labelsize': 19, 'xtick.labelsize': 16, 'ytick.labelsize': 16, 'legend.fontsize': 16, 'legend.title_fontsize': 16, 'savefig.dpi': 300, 'pdf.fonttype': 42, 'ps.fonttype': 42})

def save_figure(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(OUT / f'{name}.{ext}', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)

print(f'Output: {OUT}')

In [ ]:
inputs = {
    '2020': RESULTS / '2020/global/penetrazione/log_eta_20200210_20200217.txt',
    '2021': RESULTS / '2021/global/penetrazione/log_eta_20210628_20210704.txt',
}
frames = []
for year, path in inputs.items():
    values = pd.read_csv(path, header=None, names=['log_eta'])
    values['year'] = year
    values['eta_percent'] = 100 * 10 ** values['log_eta']
    rounded = np.rint(values['log_eta']).astype(int)
    values['decade'] = rounded.where(rounded.isin([-1, -2, -3]), 99).map({-1: r'$10^{-1}$', -2: r'$10^{-2}$', -3: r'$10^{-3}$', 99: 'other'})
    frames.append(values)
penetration = pd.concat(frames, ignore_index=True)
ks = ks_2samp(penetration.query("year == '2020'")['log_eta'], penetration.query("year == '2021'")['log_eta'])
summary = penetration.groupby('year').agg(segments=('log_eta', 'size'), median_log_eta=('log_eta', 'median'), median_eta_percent=('eta_percent', 'median'), mean_eta_percent=('eta_percent', 'mean'))
summary['ks_D'], summary['ks_pvalue'] = ks.statistic, ks.pvalue
summary.to_csv(OUT / 'viasat_summary_statistics.csv')
summary.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 3.8), constrained_layout=True)
bins = np.linspace(penetration['log_eta'].min(), penetration['log_eta'].max(), 28)
sns.histplot(data=penetration, x='log_eta', hue='year', hue_order=['2020', '2021'], bins=bins, stat='density', common_norm=False, element='step', fill=True, alpha=.18, palette=PALETTE, ax=ax)
sns.kdeplot(data=penetration, x='log_eta', hue='year', hue_order=['2020', '2021'], common_norm=False, palette=PALETTE, linewidth=2, legend=False, ax=ax)
ax.set(xlabel=r'$\log_{10}(\eta)$', ylabel='Density', title='Viasat penetration distribution')
sns.despine(ax=ax)
save_figure(fig, 'viasat_distribution')

fig, ax = plt.subplots(figsize=(6.8, 3.8), constrained_layout=True)
sns.ecdfplot(data=penetration, x='log_eta', hue='year', hue_order=['2020', '2021'], palette=PALETTE, linewidth=2, ax=ax)
x20 = np.sort(penetration.query("year == '2020'")['log_eta'].to_numpy())
x21 = np.sort(penetration.query("year == '2021'")['log_eta'].to_numpy())
grid = np.sort(np.concatenate([x20, x21]))
f20 = np.searchsorted(x20, grid, side='right') / len(x20)
f21 = np.searchsorted(x21, grid, side='right') / len(x21)
idx = np.argmax(np.abs(f20 - f21))
ax.vlines(grid[idx], min(f20[idx], f21[idx]), max(f20[idx], f21[idx]), color='.25', ls='--', lw=1.2)
ax.text(.98, .12, rf'$D={ks.statistic:.3f}$' + '\n' + rf'$p={ks.pvalue:.2g}$', transform=ax.transAxes, ha='right')
ax.set(xlabel=r'$\log_{10}(\eta)$', ylabel='Cumulative fraction', title='Cumulative penetration distribution')
sns.despine(ax=ax)
save_figure(fig, 'viasat_ecdf_ks')

In [ ]:
order = [r'$10^{-1}$', r'$10^{-2}$', r'$10^{-3}$', 'other']
shares = penetration.groupby(['year', 'decade']).size().groupby(level=0).transform(lambda value: 100 * value / value.sum()).rename('share').reset_index()
shares['decade'] = pd.Categorical(shares['decade'], order, ordered=True)
fig, ax = plt.subplots(figsize=(6.8, 3.8), constrained_layout=True)
sns.barplot(data=shares, x='decade', y='share', hue='year', hue_order=['2020', '2021'], palette=PALETTE, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=2, fontsize=14)
ax.set(xlabel=r'Rounded order of $\eta$', ylabel='Road segments [%]', title='Penetration classes')
ax.legend(title='', frameon=False)
sns.despine(ax=ax)
save_figure(fig, 'viasat_decade_shares')
shares.to_csv(OUT / 'viasat_decade_shares.csv', index=False)

In [ ]:
# Primary validation: the most observed road class, following viasat_analysis.py timing logic.
campaigns = {
    '2020': ('20200210', '20200217', ROOT / 'article' / 'data' / 'viasat_20200210_20200218_timed_fluxes.csv'),
    '2021': ('20210628', '20210704', ROOT / 'article' / 'data' / 'viasat_20210628_20210705_timed_fluxes.csv'),
}
profile_rows, metric_rows = [], []
for year, (start, stop, viasat_path) in campaigns.items():
    classes = pd.read_csv(RESULTS / year / 'global' / 'penetrazione' / 'Log' / f'{start}_{stop}' / 'Log_clusters.csv')
    cids = classes.loc[classes['penetration'] == -1, 'cid'].tolist()
    viasat = pd.read_csv(viasat_path, sep=';')
    viasat['time'] = pd.to_datetime(viasat['time'])
    viasat = viasat[viasat['cid'].isin(cids)]
    empirical = viasat.groupby(viasat['time'].dt.hour)['total_fluxes'].sum().reindex(range(24), fill_value=0).astype(float)

    daily_profiles = []
    for path in sorted((ROOT / 'article' / 'data' / 'uniq').glob(f'*_uniq_{year[2:]}*.csv.corrected')):
        frame = pd.read_csv(path, sep=';')
        hours = [column for column in frame.columns if column.startswith(year)]
        simulated = frame[frame['poly_cid'].isin(cids)].groupby('poly_cid')[hours].sum().sum(axis=0)
        simulated.index = (pd.to_datetime(simulated.index) + pd.Timedelta(minutes=75)).hour
        daily_profiles.append(simulated.groupby(level=0).sum().reindex(range(24), fill_value=0))
    simulated = pd.DataFrame(daily_profiles).mean(axis=0)
    empirical_norm = empirical / empirical.sum()
    simulated_norm = simulated / simulated.sum()
    correlation = pearsonr(simulated_norm, empirical_norm).statistic
    l2_distance = np.sqrt(np.sum((simulated_norm - empirical_norm) ** 2))
    metric_rows.append({'year': year, 'segments': len(cids), 'pearson_r': correlation, 'l2_distance': l2_distance, 'simulation_peak_hour': simulated_norm.idxmax(), 'viasat_peak_hour': empirical_norm.idxmax()})
    profile_rows.extend({'year': year, 'hour': hour, 'Simulation': simulated_norm.loc[hour], 'Viasat': empirical_norm.loc[hour]} for hour in range(24))

fit_metrics = pd.DataFrame(metric_rows)
fit_profiles = pd.DataFrame(profile_rows)
fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.1), sharey=True, constrained_layout=True)
for ax, year in zip(axes, ['2020', '2021']):
    subset = fit_profiles.query('year == @year')
    row = fit_metrics.query('year == @year').iloc[0]
    ax.plot(subset['hour'], subset['Simulation'], color=PALETTE[year], marker='o', lw=2, label='Simulation')
    ax.plot(subset['hour'], subset['Viasat'], color='#303030', marker='s', lw=1.8, ls='--', label='Viasat')
    ax.set(title=f'{year} (n={int(row.segments)})', xlabel='Hour of day', ylabel='Normalized hourly flow' if ax is axes[0] else '')
    ax.set_xticks(range(0, 24, 4))
    ax.text(.04, .93, rf'$r={row.pearson_r:.3f}$' + '\n' + rf'$L_2={row.l2_distance:.3f}$', transform=ax.transAxes, va='top')
    sns.despine(ax=ax)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(.5, -.04), ncol=2, frameon=False)
save_figure(fig, 'viasat_high_penetration_fit')
fit_metrics.to_csv(OUT / 'viasat_high_penetration_fit_metrics.csv', index=False)
fit_profiles.to_csv(OUT / 'viasat_high_penetration_fit_profiles.csv', index=False)
fit_metrics

In [ ]:
map_files = {
    '2020': RESULTS / '2020/global/penetrazione/Log/20200210_20200217/Log_clusters.gpkg',
    '2021': RESULTS / '2021/global/penetrazione/Log/20210628_20210704/Log_clusters.gpkg',
}
class_colors = {-1: '#D55E00', -2: '#E69F00', -3: '#0072B2'}
fig, axes = plt.subplots(1, 2, figsize=(11.2, 5.1), constrained_layout=True)
for ax, (year, path) in zip(axes, map_files.items()):
    roads = gpd.read_file(path)
    roads.plot(ax=ax, color='#d9d9d9', linewidth=.55)
    for decade, color in class_colors.items():
        roads.loc[roads['penetration'] == decade].plot(ax=ax, color=color, linewidth=1.1)
    ax.set_title(f'Campaign {year}')
    ax.set_axis_off()
handles = [Line2D([0], [0], color=color, lw=2.5, label=rf'$10^{{{decade}}}$') for decade, color in class_colors.items()]
fig.legend(handles=handles, loc='lower center', ncol=3, title=r'Class of $\eta$', frameon=False)
fig.suptitle('Spatial distribution of penetration classes', y=1.01)
save_figure(fig, 'viasat_spatial_classes')
print('Created figures:', sorted(p.name for p in OUT.glob('*.pdf')))